# EEP 595, Introduction to Privacy Engineering

## Summer 2024

## Lab 1: Data Anonymization with Python

##### Installation:

##### This lab assignment contains 7 questions.
Six questions are coding questions. <br>
The last question should be answered in words.

##### Answer format

You can use the coding cells for your computations.\
Enter the answer obtained in the final markdown cell in this notebook.\
Numeric values can be rounded off upto 2 decimal places.

##### Grading rubric: for each question,
100% of the points - Correct code, correct output<br>
50% of the points - Minor logical error, partial output<br>
0% of the points - No attempt, incomplete code, wrong output

##### Submission instructions

You will have to submit the completed jupyter notebook file (.ipynb) in Canvas. <br>
Please rename the submission file in the 'FirstName-LastName-Lab1.ipynb'format.

##### Acknowledgement

This lab assignment was inspired and modified from three online resources:

1. Carl McBride Ellis - Data Anonymization Using Fakes (Titanic example), available here:
https://www.kaggle.com/code/carlmcbrideellis/data-anonymization-using-faker-titanic-example

2. Florian Rohrer - A Simple Way To Anonymize Data With Pandas, available here: https://dev.to/r0f1/a-simple-way-to-anonymize-data-with-python-and-pandas-79g

Data source: Titanic - Machine Learning from Disaster, available here: https://www.kaggle.com/competitions/titanic/data?select=test.csv

3. DataCamp, Creating Synthetic Data with Python Faker Tutorial, available here: https://www.datacamp.com/tutorial/creating-synthetic-data-with-python-faker-tutorial

***
***

Importing libraries and functions

In [16]:
! pip install -q Faker

In [17]:
import pandas as pd
import numpy as np
import scipy.stats
%matplotlib inline
import matplotlib.pyplot as plt

We will be using the 'Titanic' data set for this Lab assignment. \
\
This dataset contains information about 418 unique Titanic passangers. \
\
Contents:
There are 11 variables:

- PasengerID - passenger's ID
- Pclass - a proxy for socio-economic status (SES), with values: 1st - upper, 2nd - middle, and 3rd - lower
- Name - passenger's name
- Sex - passenger's gender, with values 'male' and 'female'
- Age - passenger's age
- SibSp - information about passenger's family relations, defined as:
  * Sibling = brother, sister, stepbrother, stepsister
  * Spouse = husband, wife (mistresses and fiancés were ignored)
- ParCh: information about passenger's family relations, defined as:
 * Parent = mother, father
 * Child = daughter, son, stepdaughter, stepson

Some children travelled only with a nanny, therefore parch=0 for them.

- Ticket - passenger's ticket
- Fare - passenger's fare
- Cabin - passenger's cabin

- Embarked - information about the port of embarkation, with values:
  * C = Cherbourg,
  * Q = Queenstown,
  * S = Southampton


In [18]:
passengers = pd.read_csv('/test.csv')

In [5]:
passengers.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [6]:
passengers.head(11)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
5,897,3,"Svensson, Mr. Johan Cervin",male,14.0,0,0,7538,9.2250,NaN,S
6,898,3,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q
7,899,2,"Caldwell, Mr. Albert Francis",male,26.0,1,1,248738,29.0000,NaN,S
8,900,3,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",female,18.0,0,0,2657,7.2292,NaN,C
9,901,3,"Davies, Mr. John Samuel",male,21.0,2,0,A/4 48871,24.1500,NaN,S


Now that we have loaded the data, we are going to strip all the personally identifieable information. The columns ["PassengerId", "Name"] contain such information. Notice that ["PassengerId", "Name"] are unique for every row, so if we build a machine learning model, we would drop them anyways later on. Similar arguments can be made about ["Ticket", "Cabin"], which are almost unique for every row.

We will replace column "Name" with fake names using Faker. We will use either male of female names based on the Sex column data.

In [13]:
from faker import Faker
fake = Faker()

def replace_name_based_on_gender(row):
    if row['Sex'] == 'female':
        new_name = fake.name_female()
    else:
        new_name = fake.name_male()
    return new_name

passengers['Name'] = passengers.apply(replace_name_based_on_gender, axis=1)

# take a quick look
passengers.head(10)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,Timothy Wagner,male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,Jessica Marquez,female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,Michael Perkins,male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,Paul Brown,male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,Linda Hunter,female,22.0,1,1,3101298,12.2875,NaN,S
5,897,3,Michael Henry,male,14.0,0,0,7538,9.2250,NaN,S
6,898,3,Elizabeth Alvarado,female,30.0,0,0,330972,7.6292,NaN,Q
7,899,2,Randall Hurley,male,26.0,1,1,248738,29.0000,NaN,S
8,900,3,Janice Haynes,female,18.0,0,0,2657,7.2292,NaN,C
9,901,3,Anthony Banks,male,21.0,2,0,A/4 48871,24.1500,NaN,S


##### Question 1:  ( 2 points )

Using method 'replace_passenger_ID()', provided below please replace each passenger's unique ID with a randomly generated numerical value:

In [35]:
## Question 1 (Original)
import random

def replace_passenger_ID(row):
    return random.randint(100, 700)
# I had to expand the range since there were more than 400 people,
# I did not want any duplicte IDs to be assigned.

# Generate a set of unique random IDs
num_passengers = len(passengers)
unique_ids = set()

# Generate unique random IDs
while len(unique_ids) < num_passengers:
    unique_ids.add(replace_passenger_ID(None))

# Convert the set to a list
unique_ids = list(unique_ids)

# Shuffle the list for randomness
random.shuffle(unique_ids)

# Assign these unique IDs to the PassengerId column
passengers['PassengerId'] = unique_ids

# Save the updated data
passengers.to_csv('/test.csv', index=False)

# take a quick look
passengers.head(418)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,246,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,514,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,619,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,592,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,293,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,671,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,624,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,648,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,546,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


##### Question 2:  ( 2 points )

How many female passengers embarked from Queenstown?

In [38]:
## Question 2
# Filter for females who embarked from Queenstown
filtered_passengers = passengers[(passengers.iloc[:, 3] == 'female') & (passengers.iloc[:, 10] == 'Q')]

# Count the passengers
count_female_embarked_Q = len(filtered_passengers)
print(f"Number of female passengers from Queenstown: {count_female_embarked_Q}")


Number of female passengers from Queenstown: 24


##### Question 3: ( 2 points )

How many passengers were younger than 25 years?

In [40]:
## Question 3
# Filter for passengers under 25 years old
filtered_passengers_under_25 = passengers[passengers.iloc[:, 4] < 25]

# Count the passengers
count_passengers_under_25 = len(filtered_passengers_under_25)
print(f"Number of passengers under 25 years old: {count_passengers_under_25}")

'''
Some values in the age column were empty so number may not be accurate
historically, only accurate per the .CSV file.
'''


Number of passengers under 25 years old: 131


##### Question 4: ( 2 points )
How many passengers have traveled alone (no siblings, no spouse, no parents, no children)?

In [41]:
## Question 4
# Filter  for passengers who traveled alone (both columns F and G are 0)
travelled_alone = passengers[(passengers.iloc[:, 5] == 0) & (passengers.iloc[:, 6] == 0)]

# Count the passengers
count_travelled_alone = len(travelled_alone)
print(f"Number of passengers who traveled alone: {count_travelled_alone}")


Number of passengers who traveled alone: 253


In the next part of this lab assignment, we will create our own synthetic data set using Faker, and then, we will anonymize some of the data records in the given data set. Here is an example:
   


In [47]:
from random import randint
import pandas as pd

fake = Faker()

def generate_synthetic_data(numRecords):

    # pandas dataframe
    data = pd.DataFrame()
    for i in range(0, numRecords):
        data.loc[i,'id']= randint(1, 999)
        data.loc[i,'name']= fake.name()
        data.loc[i,'email'] = fake.email()
        data.loc[i, 'phone_number'] = fake.basic_phone_number()
        data.loc[i,'address']= fake.address()
    return data


generate_synthetic_data(25)

,id,name,email,phone_number,address
0,462.0,Jeremy Rodriguez,donaldburke@example.com,(982)313-6322,"032 Thompson Highway Suite 331\nJohnsonmouth, ..."
1,16.0,Anna Hunter,heathercraig@example.org,514-209-5621,"87300 Vasquez Tunnel\nCharleshaven, WA 11648"
2,165.0,Michael James,jeffreyrodriguez@example.net,555-845-5609,"22187 Scott Crossing\nGregoryburgh, OR 26068"
3,989.0,Ronald Kelly,rperry@example.net,3454761026,"593 Joanna Island Apt. 334\nNew Jessehaven, WI..."
4,613.0,Julie Johnson,robertsmatthew@example.net,7704785375,"36933 Stephanie Groves\nPort Benjaminville, CA..."
5,996.0,Robert Haynes,xkim@example.net,(356)683-6712,"529 Jennifer Viaduct Suite 268\nOrtizside, MH ..."
6,751.0,Michael Garcia,vpeck@example.com,828-511-1839,"671 Ortiz Crossing Apt. 599\nSouth Paul, OR 97492"
7,75.0,Zachary Reese,hughesemily@example.org,6289508420,"8302 Craig Curve\nKristiechester, AK 10042"
8,8.0,Stephanie Doyle,joshuabolton@example.com,331-756-9388,"6480 Christina Freeway\nEast Monica, NE 71655"
9,79.0,Paula Bond,jessica00@example.com,6122457163,"436 Mitchell Corner\nHamiltonport, MD 13771"


##### Question 5: ( 1 point )
Looking at the example data set, please comment on any unusual occurances your are noticing with the synthetic data.

### YOUR ANSWER HERE:

From my observations:


*   Area code in the phone numbers may or may not be valid, for and even if
they are valid, they won't match the address.
*   All email accounts are in the 'example' domain, but end in .org, .com, or
.net.
*   Not that this is wrong, but the phone number format is not consistent, but
makes the data set look more realistic.
*   The town names are based on people's names (South Toni, Tammyshire, Jacobshire, Ambertown)
* Some state acronyms aren't real, and the zip codes are made up or don't match the state.




Question 6: ( 8 points )

Using Faker, write your own code to generate 50 data records containing the following information (columns) for every record:

- Customer unique ID,
- Customer's name,
- Customer's date of birth,
- Customer's phone number,
- Customer's address,
- Customer's credit card number,
- Customer's credit card expiration date, and
- Customer's credit card securit code


In [52]:
from random import randint
import pandas as pd

fake = Faker()
numCustomers = 50;

def generate_customer_data(numCustomers):

    # Create an empty DataFrame
    data = pd.DataFrame()

    for i in range(numCustomers):
        data.loc[i, 'UniqueID'] = randint(10000, 99999)  # Unique ID
        data.loc[i, 'Name'] = fake.name()  # Customer's name
        data.loc[i, 'DoB'] = fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat()  # Date of birth
        data.loc[i, 'Phone #'] = fake.phone_number()  # Phone number
        data.loc[i, 'Address'] = fake.address().replace("\n", ", ")  # Address
        data.loc[i, 'CreditCard (CC) #'] = fake.credit_card_number(card_type="visa")  # Credit card number
        data.loc[i, 'CC Expiration'] = fake.credit_card_expire()  # Credit card expiration date
        data.loc[i, 'CC SecurityCode'] = fake.credit_card_security_code(card_type="visa")  # Credit card security code

    # Save the data
    data.to_csv('customer_data.csv', index=False)

    # Display the data
    print(data.head(10))

# Generate the r data
generate_customer_data(50)


   UniqueID                 Name         DoB                Phone #  \
0   58516.0       Audrey Mathews  1990-01-01  001-293-263-3549x4677   
1   64555.0       Dennis Johnson  2003-01-04     (477)656-0555x9265   
2   45270.0       Amber Robinson  1997-12-14     228.492.5840x86040   
3   91993.0        Alex Ferguson  1943-10-13   001-692-824-5793x622   
4   48139.0         Lauren Smith  1988-02-01  001-968-691-1369x9917   
5   63247.0    Thomas Cunningham  1950-12-07    (422)716-0407x21597   
6   58373.0        James Mccarty  2001-04-14     591.317.3996x37304   
7   88862.0  Nicholas Strickland  1945-03-03     350-557-3213x42088   
8   34531.0          Crystal Ray  1986-10-14          (842)471-1261   
9   57237.0        Linda Charles  2006-03-21             9382853052   

                                             Address CreditCard (CC) #  \
0       382 Stephanie Islands, West Edward, KY 41904  4814341397252277   
1          129 Booth Grove, South Kimberly, MD 13393  4356371623073860

Question 7: ( 8 points )

Using Faker, write your own code to generate 250 data records containing the following information (columns) for every record:

- Employee's unique ID,
- Employee's name,
- Emoloyee's date of birth,
- Employee's SSN,
- Employee's email,
- Employee's job, and
- Employee's date of last promotion.

Store your generated data into a file "Employee.csv", which you will have to submit with your notebook.
(Hint: you may find the code snippet below helpful).

In [53]:
Biodata = {'Name': ['Daniel', 'Emily', 'Sam', 'Andrea'],
        'Age': [28, 23, 35, 31],
        'Gender': ['M', 'F', 'M', 'F']
        }
df = pd.DataFrame(Biodata)

# Save the dataframe to a CSV file
df.to_csv('Biodata.csv', index=False)

In [57]:
def generate_employee_data(num_records):
    # Create an empty DataFrame
    data = pd.DataFrame()

    for i in range(num_records):
        data.loc[i, 'EmployeeID'] = randint(10000, 99999)  # Unique ID
        data.loc[i, 'Name'] = fake.name()  # Employee's name
        data.loc[i, 'DoB'] = fake.date_of_birth(minimum_age=18, maximum_age=65).isoformat()  # Date of birth
        data.loc[i, 'SSN'] = fake.ssn()  # SSN
        data.loc[i, 'Email'] = fake.email()  # Email
        data.loc[i, 'Job'] = fake.job()  # Job
        data.loc[i, 'DateOfLastPromotion'] = fake.date_between(start_date='-5y', end_date='today').isoformat()  # Date of last promotion

    # Save data to a CSV file
    data.to_csv('Employee.csv', index=False)

    # Display data
    print(data.head(10))

# Generate the data
generate_employee_data(250)


   EmployeeID                Name         DoB          SSN  \
0     98296.0       Gabriel Green  1968-11-28  222-57-8602   
1     95206.0        Donald Brown  1993-06-13  465-61-3332   
2     17628.0  Jordan Blankenship  1973-08-14  224-55-6011   
3     25412.0         Lucas Jones  1967-10-02  535-73-0002   
4     44947.0    Anthony Mcdaniel  1981-01-02  556-64-5995   
5     93520.0       Lauren Carter  1990-03-30  560-42-5678   
6     56857.0    Whitney Chambers  1976-11-06  423-38-3398   
7     54993.0      Michelle White  1976-03-22  081-55-8130   
8     26857.0        Hayden Clark  1997-09-28  418-44-1847   
9     71364.0      Jordan Mathews  1974-10-28  788-91-3970   

                       Email                                Job  \
0         smcgee@example.net  Research officer, political party   
1         aweiss@example.net                        Pathologist   
2       joshua02@example.org                Animal nutritionist   
3       uramirez@example.net                    E

In [58]:
from google.colab import files
files.download('Employee.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>